# Age Group Detection: Inference Demo

# Notebook Setup
In this section all cells required to test the coursework submission are included.

### Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

GOOGLE_DRIVE_PATH_AFTER_MYDRIVE = 'CW_Folder_UG'

GOOGLE_DRIVE_PATH = os.path.join('drive', 'My Drive', GOOGLE_DRIVE_PATH_AFTER_MYDRIVE)
print('CW folder contents:', os.listdir(GOOGLE_DRIVE_PATH))

### Install and Load Packages

All required packages are loaded here.  MTCNN is installed via pip
(used in the Week 8 tutorial for face detection).

In [ ]:
# MTCNN – used in Week 8 tutorial: 'from mtcnn import MTCNN'
!pip install mtcnn -q

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches   

from PIL import Image

# PyTorch 
import torch
import torch.nn as nn
from torchvision import models, transforms

# MTCNN face detector 
from mtcnn import MTCNN

%matplotlib inline
print('All packages loaded.')

### Load Best Model

Load the CNN (ResNet18) and the MTCNN face detector **once**, outside
`AgeDetection`, so they are not reloaded on every function call.

In [ ]:
# ── Constants ──────────────────────────────────────────────────────────────────
CLASS_NAMES = ['Child', 'Young', 'Middle-Aged', 'Senior']
MODELS_PATH = os.path.join(GOOGLE_DRIVE_PATH, 'Models')

# ImageNet normalisation values – required for ResNet18
DATA_MEANS = [0.485, 0.456, 0.406]
DATA_STDS  = [0.229, 0.224, 0.225]

# Device setup 
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# ── Load CNN (ResNet18) ────────────────────────────────────────────────────────
# Adapted from Week 8 tutorial transfer-learning example.
# Architecture must match train_cnn.ipynb exactly:
#   FC head = Dropout(0.5) → Linear(512, 4)
cnn_model = models.resnet18(weights=None)
num_ftrs   = cnn_model.fc.in_features
cnn_model.fc = nn.Sequential(
    nn.Dropout(p=0.5),
    nn.Linear(num_ftrs, len(CLASS_NAMES))
)
cnn_model.load_state_dict(
    torch.load(os.path.join(MODELS_PATH, 'cnn_resnet18.pth'), map_location=device)
)
cnn_model = cnn_model.to(device)
cnn_model.eval()   # set to inference mode
print('CNN (ResNet18) loaded.')

# ── Pre-processing transform (test-time, no augmentation) ─────────────────────
preprocess = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(DATA_MEANS, DATA_STDS)
])

# ── MTCNN face detector ────────────────────────────────────────────────────────
#   from mtcnn import MTCNN
#   mtcnn = MTCNN()
#   faces_MTCNN = mtcnn.detect_faces(img)
face_detector = MTCNN()
print('MTCNN face detector loaded.')

# Test AgeDetection



In [ ]:
def _predict_age(face_crop_rgb):
    """
    Predict the age group of a single face crop.

    Args:
        face_crop_rgb (np.ndarray): H x W x 3 uint8 RGB face image.

    Returns:
        (int, float): predicted class index and confidence.

   
    """
    face_pil    = Image.fromarray(face_crop_rgb)
    face_tensor = preprocess(face_pil).unsqueeze(0).to(device)  # [1,3,128,128]

    with torch.no_grad():                          
        output     = cnn_model(face_tensor)
        probs      = torch.softmax(output, dim=1)[0].cpu().numpy()
        pred_class = int(np.argmax(probs))

    return pred_class, float(probs[pred_class])

In [ ]:
def AgeDetection(path):
    """
    Load 4 random images from `path`, detect faces using MTCNN, predict
    the age group of each face with the CNN (ResNet18), and display the
    results with bounding boxes and age-group labels.

    Args:
        path (str): path to a folder containing image files.

    References:
        matplotlib.patches.Rectangle for bounding boxes
              'ax.add_patch(patches.Rectangle(xy=(face[0], face[1]),
               width=face[2], height=face[3], fill=False, color=..., linewidth=2))'
        MTCNN face detection
              'faces_MTCNN = mtcnn.detect_faces(img)'
              'for face in faces_MTCNN: ax.add_patch(patches.Rectangle(...))'
         PyTorch no_grad inference.
        - Zhang et al. (2016), MTCNN paper.
    """
    # ── 1. Select 4 random images from the folder ─────────────────────────────
    valid_ext = ('.jpg', '.jpeg', '.png', '.bmp')
    all_files = [
        f for f in os.listdir(path)
        if f.lower().endswith(valid_ext)
    ]

    if len(all_files) == 0:
        print('No image files found in:', path)
        return

    # Only 4 images are loaded each call 
    selected = random.sample(all_files, min(4, len(all_files)))
    n        = len(selected)

    fig, axes = plt.subplots(1, n, figsize=(5 * n, 6))
    if n == 1:
        axes = [axes]

    # One colour per age group (matches CLASS_NAMES order)
    COLOURS = ['royalblue', 'limegreen', 'darkorange', 'crimson']

    for ax, fname in zip(axes, selected):

        # ── 2. Load image ─────────────────────────────────────────────────────
        img_rgb = np.array(Image.open(os.path.join(path, fname)).convert('RGB'))
        ax.imshow(img_rgb)
        ax.set_axis_off()
        ax.set_title(fname, fontsize=8)

        # ── 3. Detect faces with MTCNN ────────────────────────────────────────
        #   faces_MTCNN = mtcnn.detect_faces(img)
        #   if faces_MTCNN is not None:
        #       for face in faces_MTCNN: ...
        detections = face_detector.detect_faces(img_rgb)

        if not detections:
            ax.set_title(f'{fname}\n(no face detected)', fontsize=8, color='grey')
            continue

        for det in detections:
            if det['confidence'] < 0.90:   # skip low-confidence detections
                continue

            # MTCNN box format: [x, y, width, height] 
            x, y, w, h = det['box']
            x  = max(0, x);  y  = max(0, y)
            x2 = min(img_rgb.shape[1], x + w)
            y2 = min(img_rgb.shape[0], y + h)

            # ── 4. Crop face and predict age group ────────────────────────────
            face_crop = img_rgb[y:y2, x:x2]
            if face_crop.size == 0:
                continue

            pred_class, confidence = _predict_age(face_crop)
            col        = COLOURS[pred_class]
            label_text = f'{CLASS_NAMES[pred_class]}  {confidence:.0%}'

            # ── 5. Draw bounding box  ─────────
            
            #   ax.add_patch(patches.Rectangle(xy=(face[0], face[1]),
            #       width=face[2], height=face[3],
            #       fill=False, color='r', linewidth=2))
            
            #   ax.add_patch(patches.Rectangle(xy=(face['box'][0], face['box'][1]),
            #       width=face['box'][2], height=face['box'][3],
            #       fill=False, color='b', linewidth=2))
            ax.add_patch(
                patches.Rectangle(
                    xy=(x, y), width=w, height=h,
                    fill=False, edgecolor=col, linewidth=2
                )
            )

            # ── 6. Overlay age-group label ────────────────────────────────────
            ax.text(
                x, y - 5,
                label_text,
                color='white', fontsize=11, fontweight='bold',
                bbox=dict(facecolor=col, alpha=0.85, pad=2, edgecolor='none')
            )

    plt.suptitle(
        'AgeDetection  –  In the Wild\n'
        '(ResNet18 CNN  +  MTCNN face detection)',
        fontsize=13, fontweight='bold'
    )
    plt.tight_layout()
    plt.show()


print('AgeDetection function defined and ready.')

Then, make a call to the `AgeDetection` function to see what results it produces.

In [ ]:
# Syntax for the next function is the following:
#
# AgeDetection(path_to_test)

path_to_test = os.path.join(GOOGLE_DRIVE_PATH, 'Personal_Dataset')
AgeDetection(path_to_test)